# HOME
---
## **Introdução** 
O modelo de Longevidade foi desenvolvido com o objetivo de avaliar o risco de ocorrência de um evento de óbito entre indivíduos de idades avançadas (65 a 90 anos de idade) em um horizonte de curto/médio prazo (em até 6 meses). O output do modelo é um score/probabilidade de 0 a 1 associado à um nível de risco que assume valores de 1 a 6, onde os indivíduos alocados no nível 1 apresentam risco mínimo (conforme atribuído pelo modelo) de virem a óbito em um futuro próximo enquanto aqueles alocados no nível 6 são os que o modelo julga mais vúlneráveis ao acontecimento. A utilização desse dado no contexto de Crédito no Banco Itaú-Unibanco visa mitigar o risco representado por aqueles clientes que vêm a faltar no cumprimento de suas obrigações financeiras em decorrência do evento de óbito, tornando-se mais um insumo disponível no desenvolvimento de políticas e processos de análise de risco de crédito cada vez mais assertivos. Do momento da escrita desta documentação (Julho/2026) o Modelo de Longevidade já teve sua aplicação e impacto avaliados no contexto da política de concessão de Crédito Consignado para o segmento INSS enquanto estudos de possíveis aplicações tanto na camada Polaris do Modelo de Risco de Crédito Visão Cliente quanto no futuro Gene do Aposentado seguem em processo de avaliação.

## **Papéis e responsáveis**


# DEFINIÇÃO DE PÚBLICO
---
## **ESTUDO DA POPULAÇÃO**
Como passo inicial na determinação do público de modelagem, foi realizada uma caracterização da população a nível Bureau/Brasil. Usando como origem  o Book de Público (database.table), foi selecionada uma janela temporal extensa, abrangendo os meses de 05/2024 a 09/2025, dentro da qual realizou-se a avaliação de diferentes segmentações desses indivíduos que permitiram traçar um perfil de volumetria total e de distribuição do público dentro do domínio de cada segmentação. 

Conforme é possível ver no quadro abaixo, a média mensal de CPFs únicos fica praticamente constante ao longo de todas as referências analisadas. Duas marcações de segmentação do público que foram bastante importantes na jornada de desenvolvimento do modelo foram a de indivíduo com conta ativa no Banco (correntista) vs o oposto (os não correntistas), e a de contratação do produto de Crédito Consignado. Na figura abaixo é possível verificar a distribuição da população brasileira como um todo ao longo da janela de tempo analisada bem como dentro das segmentações definidas. Para cada um dos grupos é trazido o valor da taxa de óbito em até 6 meses.

[TABELA EXCEL VOLUMETRIA POPULAÇÃO]

## **AMOSTRAGEM DO PÚBLICO DE DESNVOLVIMENTO**
Tendo à disposição as análises da população deu-se segmento ao processo de amostragem para obtenção do conjunto de dados de desenvolvimento. Foi definida a volumetria total de 1 milhão de indivíduos para essa amostra, garantindo assim um volume suficiente de dados para a aplicação do processo de modelagem. A amostra foi gerada buscando uma porporção de 50% correntistas e 50% não correntistas, intencionalmente aumentando a proporção amostral dos indivíduos para os quais o Banco possui mais informação disponível. Além disso, dentro do público correntista foram selecionados 250 mil indivíduos que adquiriram um contrato de Crédito Consignado ao longo das safras analisadas para permitir a avaliação do Modelo de Longevidade aplicado a concessão de um produto de crédito. Dentro de cada um dos grupos definidos a amostragem foi realizada de forma a espelhar a taxa de óbito observada por referência na população total.

[TABELA EXCEL VOLUMETRIA AMOSTRA]

## **HISTÓRICO DE DESENVOLVIMENTO**
O histórico de desenvolvimento do Modelo de Longevidade abrangeu os meses de Maio de 2024 a Setembro de 2025 (17 safras). A definição dessa janela levou em consideração a recência dos dados, a disponibilidade histórica de informações proveninetes de algumas fontes de dados e também a definição da variável resposta do modelo (link TARGET) que requer um período de 6 meses de maturação para determinação. Desse histórico, os 13 primeiros meses foram utilizados no treinamento do modelo enquanto os 4 meses finais foram aplicados no teste (out of time). É o conjnto de dados de teste que permite validar a capacidade final de generalização do modelo após seu desenvolvimento.


# VARIÁVEL RESPOSTA
---
## **MARCAÇÃO DO EVENTO DE ÓBITO**
Para marcação do evento de óbito na base de desenvolvimento foi utilizado o Book de Público (database.table). O Book disponibiliza um campo que traz informação sobre o status atual do indivíduo perante a Receita Federal e, dentre os valores assumidos pelo mesmo, o critério 'status_rct_federal = 3' marca os registros para os quais o órgão possui apontamento cadastral de óbito. 

Após realizada a marcação inicial, foram verificadas algumas inconsistências no conjunto de dados obtido, tais quais indivíduos que apresentavam registro de óbito em mais de uma data e indivíduos que apresentavam status de CPF ativo mesmo após uma marcação de evento de óbito em data anterior. Para tratar esses casos de incosistência foi adotada a regra de utilizar sempre a data da marcação mais antiga de óbito (ou a primeira) como padrão.

O procedimento completo de marcação do evento de óbito a partir das fontes de informação originais conforme realizado durante o processo de desenvolvimento do modelo pode ser consultado no seguinte Notebook[NOTEBOOK]

## **DEFINIÇÃO DO TARGET DO MODELO**
A definição da variável resposta do modelo buscou aproximar a tarefa de predição de óbito àquela de estimação de risco de crédito, permitindo a aplicação do mesmo arcabouço de métodos e ferramentas já bastante familiares aos cientistas de dados do Crédito PF à resolução do problema. Para tal, foi definida uma janela de tempo em meses (análoga a janela de performance para o risco de crédito) e a variável resposta do modelo definida de forma a assumir o valor 1 em caso de evento de óbito observado na janela a paritr da data de referência de entrada do indivíduo na base de dados ou o valor 0 no caso contrário. 

Foram realizados alguns experimentos para definição da quantidade exata de meses utilizados na composição janela de tempo de inferência do modelo. Conforme exposto na Figura abaixo, analisou-se a variação da taxa de óbito observada para diferentes faixas etárias ao longo de janelas abrangendo diferentes quantidades de meses. Observa-se que não foi constatada nenhuma tendência de estabilização da taxa de óbito conforme o aumento do número de meses, portanto a variável resposta foi definida com base na janela de performance na qual deseja-se discriminar o evento. No caso do Modelo de Longevidade decidiu-se então adotar a janela de 6 meses com a variável resposta do modelo assumindo o conceito de: ocorrência de óbito nos 6 meses seguintes a referência de entrada do indivíduo na base de dados.

[TABELA/GRÁFICOS ESTUDO EXCEL JANELA]

## **CARACTERIZAÇÃO DO EVENTO**


# EXPLORAÇÃO DE DADOS
---
## **PREMISSAS**
Durante o processo de Discovery do projeto foi realizada uma revisão da literatura científica existente relacionada ao mesmo domínio de problema: desenvolvimento de modelos de Machine Learning para predição de evento de óbito. Os trabalhos analisados evidenciam uma diferença significativa nos patamares de discriminação obtidos por modelos que utilizam variáveis com indicativos diretos de saúde dos indivíduos (resultados de exames laboratoriais, diagnósticos de doenças pré existentes...) quando comparados a modelos treinados sem este tipo específico de informaçãp. Tendo em vista esse fato e sabendo que os dados disponíveis para o desenvolvimento do Modelo de Longevidade no contexto do Banco Itaú-Unibanco não incluem este tipo de dado, decidiu-se direcionar a exploração das variáveis explicativas para 3 grandes grupos de dados:
    
    - Essenciais/Demográficos: que caracterizam os indivíduos de acordo com sua identidade e ambiente;
    - Padrões de consumo: tendências (e outras estatísticas) de gastos em determinadas categorias;
    - Proxies de fragilidade: indicativos indiretos de piora no estado de saúde dos indivíduos;

[LINK EXCEL REVIEW LITERATURA]

## **FONTES DE INFORMAÇÕES EXPLORADAS**
Abaixo encontram-se indicados todos os conjuntos de dados que foram avaliados no processo de definição das variáveis explicativas do modelo. Para cada um é fornecida uma breve descrição além do caminho da tabela de origem para consulta.

[TABELA DE FONTES DE INFORMAÇÃO]

## **SELEÇÃO DE VARIÁVEIS**

### - ANÁLISE DE PREENCHIMENTO
### - ANÁLISE DE VARIABILIDADE
### - ANÁLISE DE CORRELAÇÃO
### - RECURSIVE FEATURE ELIMINATION


# DESENVOLVIMENTO DO MODELO
---
## **OTIMIZAÇÃO DE HIPERPARÂMETROS**
O processo de otimização de hiperparâmetros utilizou o Optuna[LINK] como ferramenta. Abaixo encontram-se listados os parâmetros de entrada utilizados no processo de otimização bem como os hiperparâmetros finais selecionados, conforme pode-se consultar acessando o seguinte Notebook[NOTEBOOK].

Para obtenção de estimativas mais robustas de performance do modelo, o processo de otimização realiza uma validação cruzada no intuito de avaliar a métrica de performance em diferentes quebras do conjunto de dados de treinamento para cada combinação diferente de hiperparâmetros testados. Vale ressaltar que, por questões da dinâmica temporal inerente ao problema modelado e no intuito de evitar qualquer possibilidade de 'data leakage', os folds do processo de validação cruzada foram definidos de maneira a garantir que os dados utilizados para treinamento do modelo fossem anteriores aos dados utilizados na validação, em um padrão de janelas temporais crescentes.

## **TREINAMENTO DO MODELO**
Foi treinado um modelo LGBM (light gradient boosting machine) [LINK LGBM] utilizando os hiperparâmetros ótimos selecionados na etapa descrita acima. O quadro abaixo exibe as versões de cada biblioteca Python utilizada para o treinamento do modelo.

[VERSÕES DAS LIBS PYTHON]

Na figura abaixo é possível visualizar as curvas de aprendizado obtidas durante o treinamento do modelo tanto para o conjunto de dados de treinamento quanto para o de validação. 

## **AVALIAÇÃO DO MODELO NA BASE DE DESENVOLVIMENTO**

[GINIS]

[FEATURE IMPORTANCES]

[ANÁLISE DE DECIS]


## **AVALIAÇÃO DE MODELAGEM POR SUBPOP**


# DEFINIÇÃO DOS NÍVEIS DE RISCO
---
## **OBTENÇÃO DOS PONTOS DE CORTE**
## **CARACTERIZAÇÃO DOS NÍVEIS DE RISCO**
## **AVALIANDO CAPACIDADE DE ORDENAÇÃO**


# VALIDAÇÃO DO MODELO E RESULTADOS
---
## **BENCHMARKING DE VALIDAÇÃO**
## **ANÁLISES DE EXPLICABILIDADE**
## **AVALIAÇÃO NO CONTEXTO DO CONSIGNADO INSS**

# CONTROLES OBRIGATÓRIOS
---



# EXPLORAÇÃO DE DADOS
---
## **PREMISSAS**
Durante o processo de Discovery do projeto foi realizada uma revisão da literatura científica existente relacionada ao mesmo domínio de problema: desenvolvimento de modelos de Machine Learning para predição de evento de óbito. Os trabalhos analisados evidenciam uma diferença significativa nos patamares de discriminação obtidos por modelos que utilizam variáveis com indicativos diretos de saúde dos indivíduos (resultados de exames laboratoriais, diagnósticos de doenças pré existentes...) quando comparados a modelos treinados sem este tipo específico de informaçãp. Tendo em vista esse fato e sabendo que os dados disponíveis para o desenvolvimento do Modelo de Longevidade no contexto do Banco Itaú-Unibanco não incluem este tipo de dado, decidiu-se direcionar a exploração das variáveis explicativas para 3 grandes grupos de dados:
    
    - Essenciais/Demográficos: que caracterizam os indivíduos de acordo com sua identidade e ambiente;
    - Padrões de consumo: tendências (e outras estatísticas) de gastos em determinadas categorias;
    - Proxies de fragilidade: indicativos indiretos de piora no estado de saúde dos indivíduos;

[LINK EXCEL REVIEW LITERATURA]

## **FONTES DE INFORMAÇÕES EXPLORADAS**
Abaixo encontram-se indicados todos os conjuntos de dados que foram avaliados no processo de definição das variáveis explicativas do modelo. Para cada um dos conjuntos é indicado o tipo de informação contido na tabela de origem, o caminho da tabela de origem para consulta e o domínio no qual se encaixam segundo as 3 divisões definidas no tópico anterior. Além disso, foi incluída uma breve justificativa para os casos nos quais o conjunto dados não avançou da etapa de exploração para a fase seguinte de seleção de variáveis.

[TABELA DE FONTES DE INFORMAÇÃO]

[ACRESCENTAR SEÇÃO DE FEATURE ENGINEERING?]

Para prosseguir com o processo de modelagem, as fontes de informação aprovadas para seguir na sequência do processo de modelagem foram cruzadas com o público de desenvolvimento obtido conforme a descrição detalhada na seção [Deifnição de Públio](<!--LINK-->). O processo de extração de dados e cruzamento com a tabela de público pode ser consultado através do Notebook [1 - Featue extraction](<!--LINK-->).  

## **SELEÇÃO DE VARIÁVEIS**
O dataset utilizado como input para o processo de Seleção de Variáveis é composto de 746.029 linhas (volumetria da base final amostrada do público de desenvolvimento) e 1618 colunas compreendendo 1572 colunas de variáveis canditatas e 46 colunas que emglobam diferentes características úteis para segmentação e análise dos resultados além de colunas de identificação dos registros e da variável resposta. O objetivo final deste processo é a seleção de um subconjunto das 1572 variáveis candidatas garantindo que as features selecionadas forneçam ao modelo treinado informação de qualidade para inferir a variável resposta a partir dos dados  enquanto concomitantemente atendendo a certos requisitos necessários ou desejáveis à sua aplicação no treinamento do modelo final.

Devido ao papel central que a informação a respeito da idade ocupa no domínio do problema de predição de óbito, o processo de Seleção de Variáveis foi aplicado adotanto uma estratégia de coortes: o público amostrado de desenvolvimento foi dividido em 8 grupos de acordo com a segmentação correntista/não correntista e a sua faixa etária (4 quebras, abrangendo uma faixa de 5 anos cada) e cada um dos critérios do pipeline de Seleção de variáveis foi aplicado grupo a grupo individualmente. Na avaliação final de cada critério as variáveis foram mantidas no conjunto de candidatas somente quando verificado o preenchimento dos requisitos mínimos para todos os 8 grupos ao mesmo tempo.

Abaixo encontra-se descrita cada etapa do pipeline de Seleção de Variáveis aplicado no desenvolvimento do Modelo de Longevidade seguindo a ordem de aplicação.

### - AVALIAÇÃO DE PREENCHIMENTO HISTÓRICO
Pré requisito: dados disponíveis em todas as referências temporais do público de desenvolvimento, sem a existência de falhas de preenchimento em alguma safra específica

### - AVALIAÇÃO DE ESTABILIDADE TEMPORAL (IEP) 
Pré requisito: média dos valores de IEP (Índice de Estabilidade Populacional) calculado safra a safra (referentes a safra inicial) inferior a 10%

### - AVALIAÇÃO DE PREENCHIMENTO MÍNIMO
Pré requisito: percentual de valores preenchidos na coluna superior a 1%

### - AVALIAÇÃO DE VARIÂNCIA
Pré requisito: variância dos valores preenchidos na coluna superior a 0 (que os valores preenchidos não sejam todos iguais)

### - AVALIAÇÃO DE CORRELAÇÃO [^1]
Pré requisito: correlação de Spearman (em valor absoluto) entre duas variáveis inferior a 0,7 

### - DECISÃO JULGAMENTAL
Pré requisito: avaliação caso a caso, mas geralmente descartadas devido a baixa relação com o domínio do problema ou identificação de alguma característica indesejada após avaliação visual da distribuição das variáveis via ferramenta [Pytinela](<!--LINK-->) ([resultados](<!--LINK-->)) 

### - RECURSIVE FEATURE ELIMINATION (COM VALIDAÇÃO CRUZADA)
Pré requisito: seleciona o conjunto de variáveis para o qual o valor médio da métrica de performance atinge seu máximo (média sobre os dados de validação nos diferentes folds)


A Figura X mostra a estrutura do pipeline de Seleção de Variáveis aplicado evidenciando a quantidade de features remanescentes após cada etapa.

<!--FIGURA-->


## **VARIÁVEIS FINALISTAS DO MODELO**
A tabela a seguir traz a listagem das variáveis finalistas do modelo acompanhada de uma breve descrição:


[^1]: Durante a avaliação de correlação as variáveis são analisadas par a par e é necessário decidir qual das 2 variáveis do par deve ser descartada quando o valor aferido de correlação ultrapassa o critério definido. No contexto do Modelo de Longevidade foi desenvolvido um "score de qualidade" geral das variáveis que foi calculado previamente à aplicação da avaliação de correlação permitindo ordená-las relativamente umas as outras e descartar a de menor valor quando comparadas par a par. Esse "score de qualidade" é uma métrica que avalia as features univariadamente de acordo com cinco quesitos: poder preditivo (média do gini ao longo do tempo), estabilidade (IEP máximo ao longo do tempo), estabilidade da relação preditiva (desvio padrão do gini ao longo do tempo), robustez da relação preditiva (média da correlação de postos de Spearman ao longo do tempo) e a robustez preditiva nos coortes (gini do pior grupo). Um peso é atribuído a cada um dos quesitos para geração de um score final. A definição da função de escoragem pode ser consultada [AQUI](<!--LINK utils.py-->). 



# DESENVOLVIMENTO DO MODELO
---
## **PROCESSO DE MODELAGEM ITERATIVO**
Ao longo da execução de um projeto de Ciência de Dados é comum adotar um processo iterativo de desenvolvimento do modelo onde soluções intermediárias vão sendo produzidas e cada nova versão traz melhorias ou corrige direcionamentos adotados em versões anteriores (framework CRISPR-DM, por exexmplo). No desenvolvimento do Modelo de Longevidade foram produzidas 4 grandes versões de modelo (majors) e a última delas é a que foi produtizada e sobre a qual discorre essa documentação. Abaixo encontram-se listados os principais fatores que diferenciam cada grande versão de suas anteriores:

<!--FIGURA-->


## **OTIMIZAÇÃO DE HIPERPARÂMETROS**
Tendo definido o conjunto final de variáveis para compor o modelo, conforme detalhado na seção [Exploração de Daods](<!--LINK-->), deu-se seguimento ao processo de modelagem através da etapa de otimização de hiperparâmetros. Abaixo encontram-se listados os parâmetros de entrada utilizados pela ferramenta [Optuna](<!--LINK-->) durante o processo de otimização bem como os hiperparâmetros finais selecionados. 

<!--TABELA-->

Para obtenção de estimativas mais robustas de performance do modelo, o processo de otimização realiza uma validação cruzada no intuito de avaliar a métrica de performance em diferentes quebras do conjunto de dados de treinamento para cada combinação diferente de hiperparâmetros testados. Vale ressaltar que, por questões da dinâmica temporal inerente ao problema modelado e no intuito de evitar qualquer possibilidade de 'data leakage', os folds do processo de validação cruzada foram definidos de maneira a garantir que os dados utilizados para treinamento do modelo fossem anteriores aos dados utilizados na validação, em um padrão de janelas temporais crescentes, conforme sugere a literatura do tema [^2] [^3]. Ainda que o conjunto de dados de treinamento tenha sido corretamente controlado via deduplicação dos registros para que cada indivíduo fosse representado apenas uma vez no conjunto de dados completo, mitigando desta maneira o possível risco de treinar o modelo em dados futuros, a manutenção da ordenação temporal dos folds durante o processo de validação cruzada permite mitigar riscos com potencial origem no shift da distribuição de dados ou no conceito de variáveis que deterioram a performance em um regime de avaliação futuro e que podem ser mascarados quando analisados sob um proceso de validação cruzada utilizando folds definidos aleatoriamente.

O processo completo de otimização e seus outputs podem ser consultados acessando o  Notebook [X - Modelagem](<!--LINK-->).

[^2]: [https://arxiv.org/abs/2112.10078](https://arxiv.org/abs/2112.10078)
[^3]: [https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6336198](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6336198)


## **TREINAMENTO DO MODELO**
Foi treinado um modelo [LGBM](<!--LINK-->) (Light Gradient Boosting Machine) utilizando os hiperparâmetros ótimos selecionados na etapa descrita acima. Os quadros abaixo exibem as versões de cada biblioteca Python utilizada para o treinamento do modelo.

<!--VERSÕES DAS LIBS PYTHON-->

Durante o processo de treinamento foram utilizadas as safras de 04 e 05/2025 como conjunto de dados de validação além de definidas 2000 rodadas de boosting com um parâmetro de *early-stopping* de 50 rodadas a fim de mitigar possíveis riscos de overfitting. O modelo final treinado é um ensemble composto de 750 árvores e apresenta métricas de performance conforme o quadro abaixo: 

<!--GINIS-->

Na Figura X são exibidas as curvas de aprendizado obtidas durante o treinamento do modelo para as métricas de AUC (métrica de performance) e de *binary-logloss* (métrica de otimização da função de perda) tanto para o conjunto de dados de treinamento quanto para o de validação. As curvas permitem afirmar que, apesar do gap de performance do modelo observado entre os conjuntos de treino/validação/teste, o modelo demonstra um comportamento de aprendizado saudável, sem a presença de quaisquer sinais óbvios de overfit. O treinamento é interrompido na 750ª iteração quando ambas as curvas de treinamento e validação estão atingindo seu plateau e não observa-se aumento de performance no conjunto de treinamento após estagnação ou inversão da curva de validação conforme espera-se de regimes clássicos de overfit.  


## **AVALIAÇÃO DO MODELO NA BASE DE DESENVOLVIMENTO**

### - FEATURE IMPORTANCES

<!--FIGURA-->

### - PERFORMANCE DO MODELO

<!--FIGURA-->

### - DISCRIMINAÇÃO DO MODELO POR QUANTIL DO SCORE

<!--FIGURA-->


## **AVALIAÇÃO DE MODELAGEM POR SUBPOP**


# DEFINIÇÃO DOS NÍVEIS DE RISCO
---
## **OBTENÇÃO DOS PONTOS DE CORTE**
Os pontos de corte utilizados para determinação dos grupos de risco do Modelo de Longevidade foram obtidos através de um processo de otimização com o objetivo transformar o score contínuo do modelo de risco em um conjunto reduzido de grupos ordenados, contíguos e temporalmente estáveis. A descrição completa da metodologia aplicada pode ser consultada acessando sua [Documentação]<!--LINK-->.

A solução foi implementada através da função *optimal_binning_with_prebins* que pode ser consultada acessando o arquivo [utils.py](<!--LINK-->). Esta implementação combina quatro decisões centrais:

- Pré-binning ponderado, para representar a população e reduzir o domínio de busca.
- Minimização do IEP dos níveis de risco ao longo do tempo como objetivo primário, para alinhar a otimização à métrica de estabilidade desejada.
- Otimização lexicográfica, que define a minimização do IEP como objetivo primário e a estabilidade das taxas de evento de óbito como objetivo secundário.
- Programação dinâmica como técnica de otimização, para preservar a dependência monotônica e alcançar o ótimo global no espaço discretizado dos pré-bins.

A adoção desta metodologia e a implementação específica utilizada seguindo os critérios acima produz uma solução mais consistente do ponto de vista estatístico, computacional e de governança do que abordagens baseadas em cortes manuais, algoritmos gulosos (exemplo do Auto GH) ou programação dinâmica com estados excessivamente comprimidos.

## **CARACTERIZAÇÃO DOS NÍVEIS DE RISCO**

### - DISTRIBUIÇÃO DOS NÍVIES DE RISCO AO LONGO DO TEMPO

### - IEP AO LONGO DO TEMPO

### - TAXA DE ÓBITO POR NÍVEL DE RISCO AO LONGO DO TEMPO



## **AVALIANDO CAPACIDADE DE ORDENAÇÃO**


# DISCUSSÕES E RESULTADOS DO MODELO
---
## **BENCHMARKING DE VALIDAÇÃO**
## **ANÁLISES DE EXPLICABILIDADE**
## **AVALIAÇÃO NO CONTEXTO DO CONSIGNADO INSS**

# CONTROLES OBRIGATÓRIOS
---

# EXTRAÇÃO DE DADOS
---
## Base de Público

### **Público Alvo** 
O modelo tem como público alvo da escoragem os especialistas do produto de Microcrédito, que consistem em PJs que atuam no papel de intermedidores entre o banco e o cliente final na venda do produto. Estes PJs possuem cadastro junto a operação de Microcrédito e
seu rating/classificação mensal interfere seja na variabilidade de valores e prazos das ofertas que ele tem disponível para venda no mês vigente, seja na sua remuneração direta. Um breve acompanhamento da evolução da volumetria total mensal de especialistas ativos cadastrados junto a operação pode ser observada no gráfico abaixo:

(GRÁFICO VOLUMETRIA ESPEC/SSAFRA)

### **Histórico de Desenvolvimento** 

Para desenvolvimento e treino do modelo foram utilizadas as safras de contratação de julho/2023 a janeiro/2024. Já para avaliação do desempenho do modelo, foram utilizadas as safras de fevereiro/2024 e março/2024. A quantidade reduzida de referências utilizadas no processo de desenvolvimento do modelo se deve, entre outros, ao pequeno volume de dados existentes, consequência do fato de a própria operação de Microcrédito não possuir um número muito grande de especialistas cadastrados (qnd comparado a patamares que costumamos associar ao tamanho de conjuntos de dados de treinamento/teste de modelos de machine learning). Além disso, outros 2 fatores que contribuíram para o numero reduzido de referências utilizadas foram a disponibilidade do histórico de dados (não possuíamos histórico prévio a 2022) e a definição do target do modelo que, como tratado na seção específica sobre o tema (link Target), necessita acompanhar uma métrica de performance por no mínimo 3 meses a partir da referência avaliada para conseguir definir o label da variável resposta

### **Fontes de Informação** 

O público é construído de forma simples, utilizando a tabela de Gestão do Parceiro (db_source_repositoriosdedados_microcreditoanalytics_spec_01.tbma4_gestao_parceiro) onde são disponibilizadas as informações mais recentes e completas dos especialistas cadastrados junto a operação. 

Uma peculiaridade do processo é o fato de que, em alguns casos, mais de uma pessoa acaba atuando como especialista, porém atrelado ao mesmo CNPJ de cadastro. A esta configuração, deu-se o nome de PJ Ampliada, pelo fato do identificador único corresponder a um grupo de pessoas. Portanto, aqui torna-se importante salientar que, mesmo com a existëncia da configuração de PJ Ampliada, o rating do modelo é atribuído a nível de CNPJ, fazendo com que todos os subvendedores de uma mesma PJ Ampliada contribuam para o valor final obtido e que a todos seja atribuído o mesmo rating, tratando-os como uma única entidade. A chave única do púbico do modelo acaba sendo, então, o CNPJ/IDEQ3 do Especialista (ou grupo de subvendedores) e a safra a qual as demais informações do registro se referem.

### - Público Final de Desenvolvimento (amostragem)


## Variável Resposta
Para definição do conceito e marcação do **target** (ou variável resposta) utilizado no processo de modelagem do Angstrom foi feito uso da classificação pré existente do especialista. 

[Regras atribuição classificação especialista](../data/0_artifacts/target_dev_deals_orig.png) 

A tabela acima ilustra as regras utilizadas nesta classificação. Como é possível observar, os especialistas eram classificados em uma dentre as 7 categorias existentes e os critérios utilizados para tal se concentravam em refletir a inadimplência da sua carteira de clientes no curto prazo (máximo 3 meses anteriores, usando NPL e FPD). O rating do especialista, portanto, tratava-se de uma avaliação pós fato destes indicadores.

Com o objetivo de trazer um viés preditivo ao modelo desenvolvido, decidimos tentar abordar variáveis que expandissem nosso entendimento do especialista em si. Propusemos features para o modelo que procurassem refletir a qualidade da gestão que o especialista faz da própria carteira, bem como alguns dados sobre outras atividades profissionais exercidas por ele, seja na mesma área de atuação (Microcrédito) seja como sócio/proprietário de alguma PJ em qualquer outra área, e também alguns dados demográficos que pudessem caracterizar seu contexto socioeconômico.

Tomamos a decisão de modelar a probabilidade de o especialista vir a assumir o rating 'ddd' (pior rating da classificação anterior) em uma janela de até 3 meses. Temos, portanto:

... ==> target = 0
... ==> target = 1

Para marcação do target foi utilizada a base de classificacao dos especialistas, cruzada com a base de público


para utilização no presente modelo foi a performance 90 em 12 (atraso de 90 dias ou mais em um período de 12 meses). Para determinação da contratação interna de crédito imobiliário foi utilizado como indicador a presença de um número de contrato atrelado à proposta na base de propostas do produto disponível no SAS. O cruzamento da base de propostas com a base de contratação auxiliou na validação da informação. Para determinação da contratação de crédito imobiliário no mercado foram utilizadas as bases de [Contratação](https://confluence- itau.tecnologia.prod.ops.aws.cloud.ihf/x/kmP5Lw), (Performance Interna] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/x/WWP5Lw) e  do projeto [Foucault] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/pages/viewpage.action?pageId=799108541) que consolidam as informações contidas nas principais bases de público e performance (interna e mercado) que possuímos à disposição.


### - Bad Rate

## Variáveis explicativas
### - Fontes de informação exploradas
### - Pré seleção de variáveis


### - Extração de variáveis de books
Algumas variáveis mapeadas na listagem acima encontravam-se já disponíveis nos books de variáveis da NPM e, portanto, prontas para consumo em ambiente produtivo. Para estas variáveis, bastou a extração dos valores referentes aos registros da base de público, que foi efetuada através de uma query SQL de LEFT JOIN conforme pode ser visto nos arquivos X e Y. As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Extração Book NPM" na coluna "Método de extração". 

### - Síntese de variáveis a partir de outras fontes
Para a obtenção de algumas outras variáveis que decidimos utilizar na caracterização do especialista foi necessária a aplicação de algumas etapas de pré processamento antes de podermos usá-las efetivamente na modelagem. Nestes casos, ou o conceito que gostríamos de capturar teve de ser formulado através da agregação de algum dado mais bruto contido em outra tabela, ou o conceito já encontrava-se pronto porém nosso interesse era analisá-lo ao longo do tempo (usando janelas móveis, por exemplo). As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Variável Agregada" na coluna "Método de extração". Para todas estas variáveis é possível consultar AQUI as queries utilizadas em sua criação durante o desenvolvimento, e AQUI as queries que as reproduzem no processo batch de escoragem mensal. 

### - Tratamento de variáveis categóricas
Algumas variáveis categóricas tiveram de ser transformadas em numéricas para darmos cotinuidade ao processo de modelagem. Ainda que os algoritmos que envolvam árvores de decisão e seus derivados (como é o caso do Random Forest, utilizado na modelagem do Angstrom) consigam lidar bem com variáveis categóricas em sua maioria, demos preferência por trata-las manual e individualmente por serem poucas em quantidade e por termos maior controle sobre os possíveis valores finais assumidos pela feature.  


### - Análise de Variance Threshold
Foi realizado um corte das variáveis cuja distribuição de valores apresentavam pouca variância e que, portanto, não acrescentavam informação de grande valor ao modelo em desenvolvimento. Após realizar uma normalização de cada uma das features do conjunto de dados, utilizamos o VarianceThreshold da biblioteca Python scikit-learn e eliminamos as variáveis com variância inferior ao threshold definido de 0.02.


### - Análise visual da distribuição das features
Utilizamos o script do [Pytinela](), que acompanha a estabilidade populacional de uma feature e sua distribuição ao longo das diferentes safras analisadas. Variáveis que apresentam uma alta variabilidade populacional ao longo do tempo tendem a não ser muito indicadas para utilização em modelos preditivos: estes dependem da identificação de padrões nos dados de treino e posterior generalização do aprendizado para o conjunto de dados de teste. Caso a distribuição dos valores de uma feature varie de maneira muito brusca ou sem nenhum padrão discernível, ela pode acabar introduzindo "barulho" nos dados de treino ou até mesmo interagindo com alguma das outras variáveis preditivas, prejudicando a qualidade do modelo final. A partir desta análise acabamos excluíndo mais X variáveis do conjunto final.


### - Análise de correlação entre as variáveis explicativas
Uma outra análise que realizamos com o objetivo de selecionar as variáveis preditivas finais do modelo foi a avaliação do coeficiente de correlação de Pearson para cada par de variáveis candidatas. Os pares de variáveis que apresentaram valor do coeficiente de correlação maior que o threshold de 0.9 foram isolados e, dentre elas, foi selecionada a feature com maior valor de correlação com o target do modelo para permanecer no conjunto final. Nesta etapa, excluímos do conjunto final mais X variáveis, 
restando agora Y variáveis no total.


### - Variáveis finalistas para modelagem
Finalmente, concluídos os passos de pré seleção das variáveis explicativas, obtivemos um conjunto de XYZ variáveis totais, com as quais prosseguimos para a etapa seguinte do processo: a etapa de modelagem. Vale salientar que estas variáveis não necessariamente representam o conjunto de variáveis do modelo final, porém este é um subconjunto daquelas. Mais a frente, conforme detalhado na seção onde discutimos o processo de [Modelagem](), algumas variáveis deste conjunto acabam sendo descartadas após a avaliação do valor de Feature Importance que lhes é atribuído após o ajuste dos dados de treino ao modelo. 


### - Geração da Flat Table de Modelagem
Para as XYZ variáveis pré selecionadas, desenvolvemos um pipeline de extração e cruzamento com o público final para obtermos a flat table que serve de entrada para o processo de modelagem. O passo a passo do processo, que consiste em cruzamentos (JOINs SQL) da tabela de público com as devidas tabelas origem de cada dado, pode ser consultado [neste Jupyter Notebook](). 




# MODELAGEM
---
## Treinamento do modelo
### - Testes iniciais de modelagem
Com a [flat table de modelagem]() em mãos, realizamos alguns experimentos de modelagem utilizando métodos estatísticos e/ou algoritmos de Machine Learning que avaliamos pertinentes ao contexto do problema e ao conjunto de dados que tínhamos em mãos. 

Um dos primeiros testes que realizamos foi a utilização de um emsemble de árvores de decisão na tentativa de modelar os nossos dados. Mais especificamente, testamos a implementação da biblioteca Python [LightGBM](), em razão de já possuirmos bastante familiaridade com sua API e tmabém por já termos grande parte dos pipelines de treinamento e avaliação dos modelos criados a partir desta ferramenta bastante desenvolvidos e maduros em nossa equipe. Os experimentos, porém, não geraram resultados muito positivos: pudemos observar que, mesmo após o ajuste de hiperparâmetros, o modelo tendia a gerar um "overfit" bastante pronunciado dos dados de treino, conforme visto na curva de aprendizagem da Figura XX. A Figura mostra a evolução da função de custo nos dados de teste e validação do modelo após cada iteração do algoritmo (ou, neste caso, após a adição de cada árvore ao emsemble) e nos permite observar que, mesmo após a curva referente aos dados de validação já ter atingido seu mínimo, a curva dos dados de teste continua sua tendência decrescente indefinidamente. Este resultado nos mostra que a quantidade ideal de árvores no emsemble deveria ser YY (valor onde o mínimo da curva dos dados de validação é obtido) mas este valor além de se tratar de um número baixo de árvores também está muito distante do valor da função de custo obtida pelos dados de treinamento, nos levando a concluir, portante, que o algoritmo de Gradient Boosting é complexo demais (ou introduz muita variância) para modelar o nosso conjunto de dados que, como já falamos algumas vezes, possui poucas observações. Partimos então para experimentos com algoritmos mais simples, com menor quantidade de hiperparâmetros e complexidade no geral.

No outro extremo do espectro dos modelos de árvore, fomos testar as Árvores de Decisão simples. Construímos diversas árvores de decisão controlando, a princípio, a quantidade de níveis de profundidade ou a quantidade de folhas finais da árvore. Destes experimentos conseguimos observar que uma profundidade de 4 nívies, ou uma árvore com um máximo de 16 folhas, conseguia produzir valores de Gini que nos indicavam uma boa capacidade de discriminação (alto valor de Gini no conjunto de dados de validação) porém sem o "overfit" do conjunto de dos que observamos com o LGBM, evidenciado pela pequena diferença entre os valores de Gini para o conjunto de treino e de validação. Foram testadas algumas diferentes combinações de árvores e um apanhado dos resultados pode ser observado na Tabela XX.

Procurando um meio termo entre os emsembles gerados pelo método de Gradient Boosting (LGBM) e as Árvores de Decisão clássicas, decidimos testar o algoritmo de Random Forest, obtendo resultados bastante positivos desde o princípio. Alto poder de discriminação aliado a baixos indícios de "overfit" fizeram com a técnica acabasse sendo selecionada para gerar nosso modelo final. 



### - Seleção do modelo final
Tendo decidido pela utilização do algoritmo de Random Forest, partimos para as etapas finais de modelagem que consistem na otimização dos hiperparâmetros e na obtenção do modelo treinado. Utilizamos a ferramenta [Optuna]() para realizar o processo de otimização e os parâmetros obtidos para o treinamento da versão final do modelo estão expostos na Tabela XY a seguir.

TABELA XY

Vale destacar aqui alguns detalhes do processo de otimização. Primeiramente, o espaço amostral que definimos para exploração dos hiperparâmetros, que pode ser consultado na tabela XZ. A tabela contém o nome de cada um dos parâmetros do algoritmo Random Forest que selecionamos bem como o domínio de valores que definimos para cada um deles. O algoritmo de otimização realiza diversas iterações de amostragem destes parâmetros e para cada conjunto de valores amostrados é treinado um modelo e avaliada sua performance, na busca de maximizar (ou minimizar) uma determinada função objetivo.

Em segundo lugar, o algoritmo de amostragem escolhido, que foi o "Tree-structured Parzen Estimator" ou TPE. Este algoritmo é baseado em um processo de amostragem Bayesiano, onde a cada iteração (ou cada novo passo de exploração do espaço amostral de valores para os parâmetros do modelo) o algoritmo se utiliza das informações obtidas para realizar a escolha da amostra seguinte de maneira mais direcionada a atingir o objetivo de otimização. Se compararmos este funcionamento com o de um algoritmo de amostragem totalmente aleatório, onde a cada nova iteração seria selecionado um conjunto de valores ao acaso para utilização no treinamento do modelo, se torna mais evidente a motivação por trás desta escolha e as vantagens do algoritmo TPE.

O terceiro e último detalhe se refere a função objetivo que definimos como meta para otimização. Na grande maioria das vezes a função objetivo é definida a partir de uma métrica de performance do modelo. No nosso caso e no contexto geral de risco de crédito, normalmente estamos em busca dos parâmetros e do modelo capazes de maximizar o valor do Gini ou do KS quando avaliado no conjunto de dados de validação. Por conta da baixa volumetria de dados de treinamento que possuíamos e da forte tendência que os modelos testados apresentavam de "overfittar" nosso conjunto de dados, resolvemos definir uma função objetivo múltipla ou dupla no nosso caso: buscamos os valores de parâmetros que, ao mesmo tempo, fossem capazes de maximizar o valor de Gini porém minimizando a diferença de Gini observada entre o conjunto de dados de treinamento e de validação. Desta forma, nossa intenção é equilibrar da melhor maneira possível a qualidade/performance do modelo treinado com sua capacidade de generalização em dados desconhecidos quando em ambiente produtivo. 

Uma última etapa realizada para obtermos a versão treinada final do modelo foi a retirada do conjunto de treinamento das features que apresentavam valores de Feature Importance zerados, seguido de uma nova rodada de otimização de hiperparâmetros e retreino do modelo. No detalhe abaixo estão resumidos os dados referentes ao modleo Angstrom de acordo com a versão final implantada.  



### - Avaliação do modelo nos dados de desenvolvimento
Abaixo disponibilizamos alguns dados e métricas que costumamos analisar para aferição do poder discriminatório do modelo e que também servem, ainda que de maneira indireta, como indicativo da qualidade do modelo desenvolvido.  
        
        - Valor de gini          
        - Feature importances
        - Gráfico de discriminação do score
 
## Definição dos GHs
### - Obtenção dos grupos homogêneos
### - Distribuição do público nos GHs
### - GHs ordenando inadimplencia

## Validação e Resultados
### - Prevendo o especialista 'ddd'
### - Classificando o especialista 'hhh'
### - Matrizes de cruzamento e ginis

## Controles